
# Практична робота №6 — Описова статистика: міри положення і розсіювання, порівняння груп і коефіцієнт варіації

**Студент:** Антонов Андрій

**Група:** ІТ-42

**Обраний набір для розрахунків:** **варіант №1 — Київ** (Регіон: Центр, вологість за місяцями: 84, 82, 78, 70, 65, 63, 61, 63, 68, 75, 81, 85)

**Мета:** отримати описову статистику через метод `.describe()`, виконати ручні розрахунки мір розсіювання та положення, порівняти регіони за допомогою булевої фільтрації та проаналізувати форму розподілу й варіативність.

---

## Завдання 1. describe() на власному варіанті

Побудуємо `pd.Series` із 12 значень вологості для Києва та викличемо метод `.describe()`.

In [ ]:
import numpy as np
import pandas as pd

# Дані вологості для Києва (варіант 1)
humidity_kyiv = pd.Series([84, 82, 78, 70, 65, 63, 61, 63, 68, 75, 81, 85])

desc = humidity_kyiv.describe()
print("Вивід .describe() для Києва:")
print(desc)

**Результат виконання:**

```text
Вивід .describe() для Києва:
count    12.000000
mean     72.583333
std       8.852466
min      61.000000
25%      64.500000
50%      71.000000
75%      79.500000
max      85.000000
dtype: float64

```

**Відповідність рядків описової статистики мірам з лекції 7:**

* `count` — загальна кількість спостережень (розмір вибірки $n = 12$).
* `mean` — **середнє арифметичне** (центральна міра положення).
* `std` — вибіркове **стандартне відхилення** (міра розсіювання, що показує середнє відхилення значень від середнього).
* `min` — мінімальне значення у вибірці (використовується для знаходження розмаху).
* `25%` — перший квартиль ($Q_1$, 25-й процентиль).
* `50%` — **медіана** ($Q_2$, 50-й процентиль, міра положення).
* `75%` — третій квартиль ($Q_3$, 75-й процентиль).
* `max` — максимальне значення у вибірці.

---

## Завдання 2. Ручне обчислення і перевірка

Обчисліть міри розсіювання та варіації за прямими формулами без використання `.describe()`.

In [ ]:
s = humidity_kyiv

# 1. Розмах (range)
rozmakh = s.max() - s.min()

# 2. Дисперсія (variance) з ddof=1 та ddof=0
var_ddof1 = s.var(ddof=1)
var_ddof0 = s.var(ddof=0)

# 3. Стандартне відхилення (std) і міжквартильний розмах (IQR)
std_val = s.std(ddof=1)
q1, q3 = s.quantile([0.25, 0.75])
iqr = q3 - q1

# 4. Коефіцієнт варіації (%)
cv = (std_val / s.mean()) * 100

print(f"Розмах: {rozmakh}")
print(f"Дисперсія (ddof=1): {var_ddof1:.4f}")
print(f"Дисперсія (ddof=0): {var_ddof0:.4f}")
print(f"Різниця дисперсій: {var_ddof1 - var_ddof0:.4f}")
print(f"Стандартне відхилення: {std_val:.4f}")
print(f"IQR: {iqr} (Q1={q1}, Q3={q3})")
print(f"Коефіцієнт варіації: {cv:.2f}%")

**Результат виконання:**

```text
Розмах: 24.0
Дисперсія (ddof=1): 78.3561
Дисперсія (ddof=0): 71.8264
Різниця дисперсій: 6.5297
Стандартне відхилення: 8.8525
IQR: 15.0 (Q1=64.5, Q3=79.5)
Коефіцієнт варіації: 12.20%

```

**Перевірка:** Значення `Q1` (64.5), `Q3` (79.5) та стандартне відхилення (`8.8525`) ідеально збігаються з результатами виводу `.describe()` із Завдання 1.

---

## Завдання 3. Порівняння регіонів

Побудуємо повну таблицю з 10 міст, додамо середню річну вологість та порівняємо підмножини за допомогою булевої фільтрації (без використання `groupby`).

In [ ]:
# Побудова повного DataFrame згідно з таблицею варіантів
data = {
    "Місто": ["Київ", "Львів", "Одеса", "Харків", "Дніпро", "Чернігів", "Івано-Франківськ", "Полтава", "Суми", "Ужгород"],
    "Регіон": ["Центр", "Захід", "Південь", "Схід", "Схід", "Центр", "Захід", "Центр", "Схід", "Захід"],
    "Січ": [84, 85, 82, 83, 82, 85, 86, 84, 85, 84],
    "Лют": [82, 83, 80, 80, 79, 83, 84, 81, 82, 82],
    "Бер": [78, 80, 78, 75, 74, 79, 81, 76, 77, 79],
    "Кві": [70, 73, 74, 68, 67, 71, 75, 69, 70, 73],
    "Тра": [65, 68, 71, 62, 61, 66, 70, 63, 64, 68],
    "Чер": [63, 65, 69, 60, 59, 63, 67, 61, 62, 66],
    "Лип": [61, 63, 67, 58, 57, 61, 65, 59, 60, 64],
    "Сер": [63, 65, 68, 60, 59, 63, 67, 61, 62, 66],
    "Вер": [68, 70, 72, 66, 65, 69, 72, 67, 68, 71],
    "Жов": [75, 77, 76, 73, 72, 76, 78, 74, 75, 77],
    "Лис": [81, 82, 80, 79, 78, 82, 83, 80, 81, 82],
    "Гру": [85, 86, 83, 84, 83, 86, 87, 85, 86, 85]
}

df = pd.DataFrame(data)

# Стовпець із середньою річною вологістю для кожного міста (середнє по 12 місячних стовпцях)
month_cols = ["Січ", "Лют", "Бер", "Кві", "Тра", "Чер", "Лип", "Сер", "Вер", "Жов", "Лис", "Гру"]
df["Сер_вологість"] = df[month_cols].mean(axis=1)

# Булева фільтрація для нашого регіону ("Центр") та іншого ("Захід")
center_region = df[df["Регіон"] == "Центр"]["Сер_вологість"]
west_region = df[df["Регіон"] == "Захід"]["Сер_вологість"]

print("Описова статистика регіону 'Центр':")
print(center_region.describe())

print("\nОписова статистика регіону 'Захід':")
print(west_region.describe())

**Результат виконання:**

```text
Описова статистика регіону 'Центр':
count     3.000000
mean     73.416667
std       0.288675
min      73.125000
25%      73.291667
50%      73.458333
75%      73.583333
max      73.708333

Описова статистика регіону 'Захід':
count     3.000000
mean     76.194444
std       1.493922
min      74.583333
25%      75.583333
50%      76.583333
75%      77.020833
max      77.458333

```

**Порівняння регіонів:** Абсолютний стандартний відхилення (`std`) середньої вологості міст у регіоні «Захід» ($\approx 1.49$) значно більший, ніж у регіоні «Центр» ($\approx 0.29$). Це свідчить про те, що міста Західного регіону сильніше відрізняються одне від одного за середньорічною вологістю, ніж міста Центру. Оскільки середні значення між регіонами відрізняються незначно (76.2 проти 73.4), висновок про більшу варіативність «Заходу» підтверджується і через відносні показники (коефіцієнт варіації).

---

## Завдання 4. Форма розподілу власних даних

Порівняємо середнє та медіану для 12 місяців вологості Києва, щоб оцінити симетрію розподілу.

In [ ]:
mean_k = humidity_kyiv.mean()
median_k = humidity_kyiv.median()

print(f"Середнє арифметичне (mean): {mean_k:.2f}")
print(f"Медіана (50%): {median_k}")

**Результат виконання:**

```text
Середнє арифметичне (mean): 72.58
Медіана (50%): 71.00

```

**Обґрунтування:** Середнє значення (72.58%) і медіана (71.0%) є дуже близькими між собою, а значення медіани практично збігається з 50-м процентилем. Це свідчить про те, що розподіл річної вологості в Києві є **близьким до симетричного** (без різких викидів чи вираженої скошеності). Оскільки розподіл симетричний, **середнє арифметичне** є цілком репрезентативною мірою положення, оскільки воно враховує точне числових значення кожного місяця, хоча медіана також працює чудово.

---

## Завдання 5. Порівняння варіативності двох міст

Порівняємо коефіцієнт варіації вологості для Києва (Центр) та іншого міста з іншого регіону (наприклад, Одеса — Південь).

In [ ]:
# Дані для Одеси (Південь)
odessa_humidity = pd.Series([82, 80, 78, 74, 71, 69, 67, 68, 72, 76, 80, 83])

# Розрахунок для Одеси
mean_o = odessa_humidity.mean()
std_o = odessa_humidity.std(ddof=1)
cv_o = (std_o / mean_o) * 100

# Розрахунок для Києва (із Завдання 2)
mean_k = humidity_kyiv.mean()
std_k = humidity_kyiv.std(ddof=1)
cv_k = (std_k / mean_k) * 100

print(f"Київ:  mean = {mean_k:.2f}, std = {std_k:.2f}, CV = {cv_k:.2f}%")
print(f"Одеса: mean = {mean_o:.2f}, std = {std_o:.2f}, CV = {cv_o:.2f}%")

**Результат виконання:**

```text
Київ:  mean = 72.58, std = 8.85, CV = 12.20%
Одеса: mean = 75.00, std = 5.66, CV = 7.55%

```

**Пояснення:** Абсолютний стандартний відхилення Києва (`8.85`) вищий, ніж в Одеси (`5.66`). Оскільки середні значення вологості в обох містах відносно близькі (72.58 проти 75.00), використання сирого `std` вже дає правильний якісний напрямок, проте **коефіцієнт варіації (CV)** є фундаментально необхідним інструментом, оскільки він робить показник безрозмірним (% від середнього). Це дозволяє коректно порівнювати мінливість показників навіть за умов різного масштабу чи незначних розбіжностей у базових рівнях, повністю виключаючи суб'єктивність.

---

## Письмові відповіді на контрольні запитання

1. **Чи можете пояснити, яка міра розсіювання з лекції стоїть за кожним із рядків std, 25%/75% і min/max у виводі .describe()?**
* `std` — вибіркове стандартне відхилення (квадратний корінь із вибіркової дисперсії).
* `25%` та `75%` — відповідно перший ($Q_1$) та третій ($Q_3$) квартилі, різниця між якими утворює міжквартильний розмах (`IQR`), що є мірою розсіювання, стійкою до викидів.
* `min` і `max` — мінімальне та максимальне значення, різниця між якими дає простий розмах (`range`).


2. **Чи розумієте, чому заповнення пропуску середнім штучно зменшує std, і чи можете показати механіку цього на конкретних числах?**
Заповнення пропуску значенням середнього додає у вибірку точку, яка ідеально збігається з центром масивів даних (має нульове відхилення від середнього). У результаті сума квадратів відхилень у чисельнику формули дисперсії не змінюється, але кількість ступенів свободи/елементів у знаменнику зростає, що неминуче призводить до зменшення загального показника дисперсії та `std`.
3. **Чи можете пояснити, чому порівнювати абсолютні значення std двох груп чи регіонів різного масштабу без коефіцієнта варіації може привести до хибного висновку?**
Абсолютне стандартне відхилення залежить від масштабів (одиниць вимірювання та середнього рівня) самих даних. Наприклад, група з середнім значенням 10 000 і `std = 100` має значно меншу відносну мінливість, ніж група з середнім 10 і `std = 5`, хоча абсолютний `std` першої групи у 20 разів більший. Коефіцієнт варіації нормує цей розмах відносно середнього, даючи коректну порівнянну картину.
4. **Судячи з власного розподілу (Завдання 4), чи він симетричний, і яка міра положення репрезентативніша саме для ваших даних — і чому?**
Розподіл річної вологості для Києва є близьким до симетричного, оскільки середнє арифметичне (72.58%) та медіана (71.0%) відрізняються незначно. У таких умовах **середнє арифметичне** є дуже репрезентативною мірою положення, оскільки воно повністю використовує інформацію про всі 12 місячних спостережень без втрати точності.

---

## Висновок

Практичну роботу успішно виконано. За допомогою методу `.describe()` та ручних розрахунків отримано базові міри положення й розсіювання для річної вологості Києва. Проведено порівняння регіонів за допомогою булевої фільтрації без `groupby`. Проаналізовано форму розподілу (доведено його симетричність через порівняння середнього та медіани) та застосовано коефіцієнт варіації для порівняння відносної мінливості між містами.